# GovMove setup
Lands every dataset GovMove needs in **one** Unity Catalog location, `<catalog>.<schema>`, and builds the app tables there.

**Run order** (see `README.md`):
1. Run this notebook.
2. Run `Government Property Sustainability and Investment Prioritisation`, which writes `property_priority_results`.
3. Run this notebook again with `overwrite_existing = false`, so that `govmove_property_evidence` includes the priority scores.

**Inputs (widgets):**
* `catalog`, `schema` (required): the landing location for all datasets and app tables. The app uses the same values: `GOVMOVE_CATALOG` and `GOVMOVE_SCHEMA` in `app.yaml`.
* `addressbase_mode`: `table` or `volume`.
  * `table` creates `<catalog>.<schema>.addressbasepremium` as a view over the existing table named in `addressbase_table` (no copy).
  * `volume` loads OS AddressBase Premium CSV files from `addressbase_volume_path`.
* `overwrite_existing`: `false` (default) leaves source tables that already exist untouched. `true` re-downloads them and replaces them.
* `priority_table` (optional): the full name of `property_priority_results` from the prioritisation notebook. It adds priority scores to the evidence table.

**Sources**
* The Government Property Estate including Buildings and Land ([data.gov.uk](https://www.data.gov.uk/dataset/a3ad62cf-d8a2-4d3b-b3c6-24c35c5e0f71/the-government-property-estate-including-buildings-and-land)), Open Government Licence. The six XLSX resources are found through the CKAN API at run time:

  | Resource | Table |
  | --- | --- |
  | Property Contract | `property` |
  | Building | `building` |
  | Land | `land` |
  | Occupation | `occupation` |
  | Disposal | `surplus` |
  | Vacant Space | `vacant_space` |
* AddressBase Premium (Ordnance Survey). This is a licensed product, so it isn't downloadable from data.gov.uk. It must already exist as a table, or be supplied as files in a volume.

**Tables created in `<catalog>.<schema>`:**
* Source tables (column names exactly as published):
  * `property`, `building`, `land`, `occupation`, `surplus`, `vacant_space`
  * `addressbasepremium`
* App objects:
  * views `govmove_property_clean`, `govmove_building_clean`, `govmove_land_clean`, `govmove_occupation_clean` and `vw_govmove_property_locations`
  * tables `govmove_property_locations`, `govmove_postcode_centroids` and `govmove_property_evidence`

Run on serverless or a cluster with Databricks Runtime 17.1+ (spatial `st_transform`).

In [0]:
%pip install -q openpyxl

In [0]:
dbutils.widgets.text("catalog", "", "1. Catalog (landing)")
dbutils.widgets.text("schema", "", "2. Schema (landing)")
dbutils.widgets.dropdown("addressbase_mode", "table", ["table", "volume"], "3. AddressBase source")
dbutils.widgets.text("addressbase_table", "", "4. AddressBase table (table mode)")
dbutils.widgets.text("addressbase_volume_path", "", "5. AddressBase CSV path (volume mode)")
dbutils.widgets.dropdown("overwrite_existing", "false", ["false", "true"], "6. Overwrite existing source tables")
dbutils.widgets.text("priority_table", "", "7. Priority results table (optional)")
dbutils.widgets.text("irish_grid_epsg", "29903", "8. Irish Grid EPSG")

import re
CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA = dbutils.widgets.get("schema").strip()
IDENT = re.compile(r"^[A-Za-z0-9_\-]+$")
FQN = re.compile(r"^[A-Za-z0-9_\-]+\.[A-Za-z0-9_\-]+\.[A-Za-z0-9_\-]+$")
assert CATALOG and SCHEMA, "Enter the landing catalog and schema in the widgets at the top of the notebook."
assert IDENT.match(CATALOG) and IDENT.match(SCHEMA), "Catalog and schema may contain only letters, digits, '_' and '-'."
LANDING = f"`{CATALOG}`.`{SCHEMA}`"
ABP_MODE = dbutils.widgets.get("addressbase_mode")
ABP_TABLE = dbutils.widgets.get("addressbase_table").strip()
ABP_PATH = dbutils.widgets.get("addressbase_volume_path").strip()
OVERWRITE = dbutils.widgets.get("overwrite_existing") == "true"
PRIORITY_TABLE = dbutils.widgets.get("priority_table").strip()
IG_EPSG = int(dbutils.widgets.get("irish_grid_epsg"))
for name, v in (("addressbase_table", ABP_TABLE), ("priority_table", PRIORITY_TABLE)):
    assert not v or FQN.match(v), f"{name} must be a full catalog.schema.table name."

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {LANDING} COMMENT 'GovMove landing schema'")

def t(name):
    """Fully qualified, quoted name of an object in the landing schema."""
    return f"{LANDING}.`{name}`"

def exists(name):
    return spark.catalog.tableExists(f"{CATALOG}.{SCHEMA}.{name}")

print(f"Landing location: {CATALOG}.{SCHEMA} | AddressBase: {ABP_MODE} | overwrite: {OVERWRITE}")

In [0]:
import io, datetime, requests, pandas as pd
from pyspark.sql import functions as F

CKAN_ID = "a3ad62cf-d8a2-4d3b-b3c6-24c35c5e0f71"
# data.gov.uk resource name -> landing table name
RESOURCES = {"Property Contract": "property", "Building": "building", "Land": "land",
             "Occupation": "occupation", "Disposal": "surplus", "Vacant Space": "vacant_space"}

# Non-string column types (everything else is STRING). Matches the published data dictionary.
TYPE_MAP = {
    "property": {"Latitude": "double", "Longitude": "double", "Easting": "bigint", "Northing": "bigint", "NI Easting": "bigint",
                 "NI Northing": "bigint", "Total Land Area": "double", "Acquisition Date": "date", "Floor Area": "double"},
    "building": {"Easting": "bigint", "Northing": "bigint", "NI Easting": "bigint", "NI Northing": "bigint", "Latitude": "double",
                 "Longitude": "double", "Floor Area": "double", "Construction Date": "date", "BREEAM Reference": "bigint",
                 "DEC Reference": "bigint", "DEC Certificate Issue Date": "date", "DEC Operating Rating Value": "bigint",
                 "DEC Expiry Date": "date", "EPC Reference": "bigint", "EPC Property Rating Value": "bigint",
                 "EPC Expiry Date": "date", "EPC Certificate Issue Date": "date"},
    "land": {"Total Land Area": "double", "Easting": "bigint", "Northing": "bigint", "NI Easting": "bigint", "NI Northing": "bigint",
             "Latitude": "double", "Longitude": "double"},
    "occupation": {"Easting": "bigint", "Northing": "bigint", "NI Easting": "bigint", "NI Northing": "bigint", "Latitude": "double",
                   "Longitude": "double", "Floor Area": "double"},
    "surplus": {"Surplus Status": "boolean", "Easting": "bigint", "Northing": "bigint", "NI Easting": "bigint", "NI Northing": "bigint",
                "Latitude": "double", "Longitude": "double", "Total Surplus Land Area": "double", "Total Surplus Floor Area": "double"},
    "vacant_space": {"Easting": "bigint", "Northing": "bigint", "NI Easting": "bigint", "NI Northing": "bigint", "Latitude": "double",
                     "Longitude": "double", "Vacant Area": "double"},
}
# Vacant-space dates are kept as 'dd/MM/yyyy HH:mm' strings (the app parses them).
DMY_COLS = {"vacant_space": ["Vacant Space Available Start Date", "Vacant Space Availability End Date", "Vacation Date"]}


def resource_urls():
    r = requests.get("https://ckan.publishing.service.gov.uk/api/action/package_show", params={"id": CKAN_ID}, timeout=120)
    r.raise_for_status()
    res = {x["name"]: x["url"] for x in r.json()["result"]["resources"] if x.get("format", "").upper() == "XLSX"}
    missing = [n for n in RESOURCES if n not in res]
    assert not missing, f"Resources not found on data.gov.uk: {missing}"
    return {RESOURCES[n]: res[n] for n in RESOURCES}


def xlsx_to_spark(table, content):
    """Read the first sheet as text (literal values such as 'N/A' preserved), then apply explicit casts."""
    pdf = pd.read_excel(io.BytesIO(content), sheet_name=0, dtype=str, engine="openpyxl", keep_default_na=False, na_values=[""])
    pdf = pdf.astype(object).where(pdf.notna(), None)
    sdf = spark.createDataFrame(pdf, ", ".join(f"`{c}` string" for c in pdf.columns))
    cols = []
    for c in pdf.columns:
        col, typ = F.col(f"`{c}`"), TYPE_MAP[table].get(c)
        if c in DMY_COLS.get(table, []):
            e = F.date_format(F.try_to_timestamp(F.trim(col), F.lit("yyyy-MM-dd HH:mm:ss")), "dd/MM/yyyy HH:mm")
        elif typ == "bigint":
            e = F.trim(col).try_cast("double").try_cast("bigint")
        elif typ == "double":
            e = F.trim(col).try_cast("double")
        elif typ == "date":
            e = F.to_date(F.substring(F.trim(col), 1, 10))
        elif typ == "boolean":
            v = F.lower(F.trim(col))
            e = F.when(v.isin("true", "yes", "1"), True).when(v.isin("false", "no", "0"), False)
        else:
            e = col
        cols.append(e.alias(c))
    return sdf.select(*cols, F.lit(None).cast("string").alias("_rescued_data"))


urls = resource_urls()
landed = {}
for table, url in urls.items():
    if exists(table) and not OVERWRITE:
        landed[table] = "kept existing"
        continue
    content = requests.get(url, timeout=600).content
    df = xlsx_to_spark(table, content)
    # Column mapping keeps the published column names (with spaces) on Delta.
    (df.write.mode("overwrite").option("overwriteSchema", "true").option("delta.columnMapping.mode", "name")
       .saveAsTable(f"{CATALOG}.{SCHEMA}.{table}"))
    stamp = datetime.datetime.utcnow().strftime("%Y-%m-%d %H:%M UTC")
    spark.sql(f"COMMENT ON TABLE {t(table)} IS 'Government Property Estate ({table}) from {url}, downloaded {stamp}. OGL.'")
    landed[table] = "downloaded"

display(spark.createDataFrame([(k, landed[k], spark.table(f"{CATALOG}.{SCHEMA}.{k}").count(), urls[k]) for k in urls],
                              "table string, action string, rows long, source string"))

In [0]:
# Columns GovMove needs from AddressBase Premium (flattened, one row per address record).
ABP_REQUIRED = {"postcode", "easting", "northing", "latitude", "longitude", "effectiveenddate", "addressstatus"}

def abp_object_type():
    if not exists("addressbasepremium"):
        return None
    return spark.sql(f"DESCRIBE TABLE EXTENDED {t('addressbasepremium')}").where("col_name = 'Type'").first()["data_type"]

current = abp_object_type()
if current and current != "VIEW" and not OVERWRITE:
    print(f"Kept existing AddressBase table {CATALOG}.{SCHEMA}.addressbasepremium ({current}).")
elif ABP_MODE == "table":
    assert ABP_TABLE, "Set addressbase_table to the full name of an existing AddressBase Premium table."
    missing = ABP_REQUIRED - {c.lower() for c in spark.table(ABP_TABLE).columns}
    assert not missing, f"{ABP_TABLE} is missing required columns: {sorted(missing)}"
    if ABP_TABLE.lower() != f"{CATALOG}.{SCHEMA}.addressbasepremium".lower():
        q = ".".join(f"`{p}`" for p in ABP_TABLE.split("."))
        spark.sql(f"CREATE OR REPLACE VIEW {t('addressbasepremium')} COMMENT 'AddressBase Premium (view over {ABP_TABLE})' AS SELECT * FROM {q}")
    print(f"addressbasepremium -> view over {ABP_TABLE}")
else:
    assert ABP_PATH.startswith("/Volumes/"), "Set addressbase_volume_path to a /Volumes/... folder of AddressBase Premium CSV files."
    abp = spark.read.option("header", "true").option("inferSchema", "false").csv(ABP_PATH)
    abp = abp.toDF(*[c.strip().lower() for c in abp.columns])
    missing = ABP_REQUIRED - set(abp.columns)
    assert not missing, f"CSV files are missing required columns: {sorted(missing)}"
    abp = (abp.withColumn("easting", F.col("easting").try_cast("float")).withColumn("northing", F.col("northing").try_cast("float"))
              .withColumn("latitude", F.col("latitude").try_cast("float")).withColumn("longitude", F.col("longitude").try_cast("float")))
    abp.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{SCHEMA}.addressbasepremium")
    print(f"Loaded AddressBase Premium from {ABP_PATH}")

In [0]:
# Snake_case views over the landed tables. Joins elsewhere use exact Insite Property Reference only.
UK_LATLON = "(CAST(`Latitude` AS DOUBLE) BETWEEN 49.0 AND 61.0 AND CAST(`Longitude` AS DOUBLE) BETWEEN -8.7 AND 2.0)"
BNG_OK = "(`Easting` > 0 AND `Easting` <= 700000 AND `Northing` > 0 AND `Northing` <= 1300000)"

def dup_flag(table):
    """True for the second and later copies of an exactly repeated source row."""
    cols = ", ".join(f"`{c}`" for c in spark.table(f"{CATALOG}.{SCHEMA}.{table}").columns if c != "_rescued_data")
    return f"ROW_NUMBER() OVER (PARTITION BY xxhash64({cols}) ORDER BY 1) > 1"

CLEAN_VIEWS = {
"govmove_property_clean": f"""
SELECT `Insite Property Reference` AS insite_property_reference, `Insite Contract Reference` AS insite_contract_reference,
       concat_ws('-', `Insite Property Reference`, `Insite Contract Reference`) AS record_reference,
       upper(trim(`Property Name`)) AS property_name_clean,
       nullif(upper(trim(concat_ws(' ', `Property Number`, `Street Name`, `Town`))), '') AS full_address_clean,
       upper(trim(`Town`)) AS town_clean, upper(trim(`Region`)) AS region_clean, upper(trim(`Local Authority`)) AS local_authority_clean,
       upper(trim(`Postcode`)) AS postcode_clean, trim(`Country`) AS uk_nation,
       `Contract Name` AS contract_name, `Organisation` AS contract_organisation, `Parent Organisation` AS contract_parent_organisation,
       `Tenure` AS tenure, `Asset Type` AS asset_type, `Usage` AS usage, CAST(`Floor Area` AS DOUBLE) AS floor_area,
       CASE WHEN {BNG_OK} THEN CAST(`Easting` AS DOUBLE) END AS easting_clean,
       CASE WHEN {BNG_OK} THEN CAST(`Northing` AS DOUBLE) END AS northing_clean,
       CASE WHEN {UK_LATLON} THEN CAST(`Latitude` AS DOUBLE) END AS latitude_clean,
       CASE WHEN {UK_LATLON} THEN CAST(`Longitude` AS DOUBLE) END AS longitude_clean,
       {UK_LATLON} AS has_valid_coordinates, {dup_flag('property')} AS is_duplicate_copy
FROM {t('property')}""",
"govmove_building_clean": f"""
SELECT `Insite Property Reference` AS insite_property_reference, `Insite Building Reference` AS insite_building_reference,
       concat_ws('-', `Insite Property Reference`, `Insite Building Reference`) AS record_reference,
       `Building Name` AS building_name, CAST(`Floor Area` AS DOUBLE) AS floor_area, `Floor Area Measurement` AS floor_area_measurement,
       `Construction Date` AS construction_date, `Usage` AS usage, `Listed Building` AS listed_building,
       coalesce(nullif(trim(`Listed Building`), ''), 'Not Listed') <> 'Not Listed' AS listed_building_flag,
       nullif(upper(trim(`EPC Property Rating`)), '') AS epc_band, nullif(upper(trim(`DEC Operational Rating`)), '') AS dec_band,
       upper(trim(`Region`)) AS region_clean, CASE WHEN {UK_LATLON} THEN CAST(`Latitude` AS DOUBLE) END AS latitude,
       {dup_flag('building')} AS is_duplicate_copy
FROM {t('building')}""",
"govmove_land_clean": f"""
SELECT `Insite Land Reference` AS insite_land_reference, substr(`Insite Land Reference`, 1, 8) AS insite_property_reference,
       `Insite Land Reference` AS record_reference, CAST(`Total Land Area` AS DOUBLE) AS total_land_area,
       CAST(`Total Land Area` AS DOUBLE) AS land_area_hectares, `Usage` AS usage, `Title Number` AS title_number,
       `Conservation Area` AS conservation_area, upper(trim(`Region`)) AS region_clean, {UK_LATLON} AS has_valid_coordinates,
       {dup_flag('land')} AS is_duplicate_copy
FROM {t('land')}""",
"govmove_occupation_clean": f"""
SELECT `InSite Occupation Ref` AS insite_occupation_reference, `Insite Property Reference` AS insite_property_reference,
       `InSite Occupation Ref` AS record_reference, `Occupation Name` AS occupation_name, `Organisation` AS organisation,
       CAST(`Floor Area` AS DOUBLE) AS floor_area, `Floor Area Measurement` AS floor_area_measurement, `Agreement Type` AS agreement_type,
       `Usage` AS usage, upper(trim(`Region`)) AS region_clean, {UK_LATLON} AS has_valid_coordinates,
       {dup_flag('occupation')} AS is_duplicate_copy
FROM {t('occupation')}""",
}
for name, sql in CLEAN_VIEWS.items():
    spark.sql(f"CREATE OR REPLACE VIEW {t(name)} AS {sql}")
    print(f"{name}: {spark.table(f'{CATALOG}.{SCHEMA}.{name}').count():,} rows")

In [0]:
# Raw string: '\\s' must reach Spark SQL unchanged.
spark.sql(rf"""
CREATE OR REPLACE VIEW {t('vw_govmove_property_locations')}
COMMENT 'GovMove: vacant-space and surplus records with coordinates normalised to British National Grid (EPSG:27700).'
AS
WITH src AS (
  SELECT 'vacant_space' AS dataset, `InSite Vacant Space Reference` AS record_reference, `Insite Property Reference` AS property_reference,
         `Property Name` AS property_name, `Property Number` AS property_number, `Street Name` AS street_name, `Town` AS town,
         `Postcode` AS postcode_raw, `Region` AS region, `Local Authority` AS local_authority,
         trim(`Vacant Space Status`) AS availability_status, CAST(`Vacant Area` AS DOUBLE) AS available_area,
         concat(coalesce(`Vacant Area Units`, 'unit not recorded'), ' (assumed sqm)') AS area_units,
         CAST(`Vacant Area` AS DOUBLE) AS available_area_sqm, CAST(NULL AS DOUBLE) AS surplus_land_area_hectares,
         to_date(try_to_timestamp(`Vacant Space Available Start Date`, 'dd/MM/yyyy HH:mm')) AS available_start_date,
         to_date(try_to_timestamp(`Vacant Space Availability End Date`, 'dd/MM/yyyy HH:mm')) AS available_end_date,
         `Security Classification` AS security_classification,
         CAST(`Easting` AS DOUBLE) AS easting_raw, CAST(`Northing` AS DOUBLE) AS northing_raw,
         CAST(`NI Easting` AS DOUBLE) AS ni_easting_raw, CAST(`NI Northing` AS DOUBLE) AS ni_northing_raw,
         CAST(`Latitude` AS DOUBLE) AS latitude_raw, CAST(`Longitude` AS DOUBLE) AS longitude_raw
  FROM {t('vacant_space')}
  UNION ALL
  SELECT 'surplus', `In Site Disposal Reference`, `Insite Property Reference`, `Property Name`, `Property Number`, `Street Name`, `Town`,
         `Postcode`, `Region`, `Local Authority`, coalesce(trim(`Status of Sale`), 'No sale status'),
         CAST(`Total Surplus Floor Area` AS DOUBLE), 'Surplus floor area (assumed sqm)', CAST(`Total Surplus Floor Area` AS DOUBLE),
         CAST(`Total Surplus Land Area` AS DOUBLE), CAST(NULL AS DATE), CAST(NULL AS DATE), `Security Classification`,
         CAST(`Easting` AS DOUBLE), CAST(`Northing` AS DOUBLE), CAST(`NI Easting` AS DOUBLE), CAST(`NI Northing` AS DOUBLE),
         CAST(`Latitude` AS DOUBLE), CAST(`Longitude` AS DOUBLE)
  FROM {t('surplus')} WHERE `Surplus Status` = true
),
flags AS (
  SELECT *,
    CASE WHEN length(regexp_replace(upper(coalesce(postcode_raw, '')), '\\s', '')) BETWEEN 5 AND 7
         THEN concat(substr(regexp_replace(upper(postcode_raw), '\\s', ''), 1, length(regexp_replace(upper(postcode_raw), '\\s', '')) - 3), ' ',
                     substr(regexp_replace(upper(postcode_raw), '\\s', ''), -3))
         ELSE nullif(regexp_replace(upper(coalesce(postcode_raw, '')), '\\s', ''), '') END AS postcode,
    (easting_raw > 0 AND northing_raw > 0 AND easting_raw <= 700000 AND northing_raw <= 1300000) AS bng_in_range,
    (ni_easting_raw > 0 AND ni_northing_raw > 0 AND ni_easting_raw <= 400000 AND ni_northing_raw <= 500000) AS ni_in_range,
    (upper(coalesce(postcode_raw, '')) LIKE 'BT%' OR (latitude_raw BETWEEN 54.0 AND 55.4 AND longitude_raw BETWEEN -8.2 AND -5.4)) AS is_ni_location
  FROM src
),
classified AS (
  SELECT *, CASE
      WHEN coalesce(bng_in_range, false) AND is_ni_location AND easting_raw > 200000 THEN 'crs_mismatch_suspected'
      WHEN coalesce(bng_in_range, false) THEN 'valid_bng'
      WHEN is_ni_location AND coalesce(ni_in_range, false) THEN 'transformed_from_irish_grid'
      WHEN coalesce(ni_in_range, false) THEN 'ni_coordinates_outside_ni'
      WHEN easting_raw IS NULL OR northing_raw IS NULL THEN 'missing'
      WHEN easting_raw = 0 OR northing_raw = 0 THEN 'zero'
      ELSE 'out_of_range' END AS coordinate_status
  FROM flags
),
projected AS (
  SELECT *,
    coordinate_status IN ('valid_bng', 'transformed_from_irish_grid') AS is_comparable,
    CASE coordinate_status WHEN 'valid_bng' THEN 'EPSG:27700'
                           WHEN 'transformed_from_irish_grid' THEN 'EPSG:{IG_EPSG} transformed to EPSG:27700' END AS source_crs,
    CASE coordinate_status WHEN 'valid_bng' THEN easting_raw
         WHEN 'transformed_from_irish_grid' THEN st_x(st_transform(st_setsrid(st_point(ni_easting_raw, ni_northing_raw), {IG_EPSG}), 27700)) END AS bng_easting,
    CASE coordinate_status WHEN 'valid_bng' THEN northing_raw
         WHEN 'transformed_from_irish_grid' THEN st_y(st_transform(st_setsrid(st_point(ni_easting_raw, ni_northing_raw), {IG_EPSG}), 27700)) END AS bng_northing,
    CASE coordinate_status
      WHEN 'valid_bng' THEN 'British National Grid Easting/Northing used directly.'
      WHEN 'transformed_from_irish_grid' THEN 'NI Easting/Northing (Irish Grid) transformed to British National Grid.'
      WHEN 'crs_mismatch_suspected' THEN 'Cannot be compared reliably: Northern Ireland location with Easting/Northing that do not look like British National Grid.'
      WHEN 'ni_coordinates_outside_ni' THEN 'Cannot be compared reliably: NI Easting/Northing recorded for a location outside Northern Ireland and no valid British National Grid values.'
      WHEN 'missing' THEN 'Cannot be compared: Easting/Northing missing.'
      WHEN 'zero' THEN 'Cannot be compared: Easting/Northing recorded as zero.'
      ELSE 'Cannot be compared: Easting/Northing outside the valid British National Grid range.' END AS coordinate_note
  FROM classified
)
SELECT dataset, record_reference, property_reference, property_name, property_number, street_name, town, postcode, postcode_raw,
       region, local_authority,
       concat_ws(', ', nullif(trim(concat_ws(' ', property_number, street_name)), ''), town, postcode) AS full_address,
       availability_status, available_area, area_units, available_area_sqm, surplus_land_area_hectares,
       available_start_date, available_end_date, security_classification,
       easting_raw, northing_raw, ni_easting_raw, ni_northing_raw, latitude_raw, longitude_raw,
       coordinate_status, is_comparable, source_crs, bng_easting, bng_northing, coordinate_note,
       CASE WHEN latitude_raw BETWEEN 49.8 AND 60.9 AND longitude_raw BETWEEN -8.7 AND 1.8 THEN latitude_raw
            WHEN is_comparable THEN st_y(st_transform(st_setsrid(st_point(bng_easting, bng_northing), 27700), 4326)) END AS map_latitude,
       CASE WHEN latitude_raw BETWEEN 49.8 AND 60.9 AND longitude_raw BETWEEN -8.7 AND 1.8 THEN longitude_raw
            WHEN is_comparable THEN st_x(st_transform(st_setsrid(st_point(bng_easting, bng_northing), 27700), 4326)) END AS map_longitude
FROM projected
""")
spark.sql(f"CREATE OR REPLACE TABLE {t('govmove_property_locations')} COMMENT 'GovMove app source, materialised from vw_govmove_property_locations' "
          f"AS SELECT * FROM {t('vw_govmove_property_locations')}")
display(spark.table(f"{CATALOG}.{SCHEMA}.govmove_property_locations").groupBy("dataset", "availability_status", "coordinate_status").count()
        .orderBy("dataset", "availability_status", "coordinate_status"))

In [0]:
# Mean BNG (EPSG:27700) Easting/Northing of AddressBase Premium address points per postcode.
# Current approved/provisional addresses preferred; historical points used only where no current address exists.
spark.sql(f"""
CREATE OR REPLACE TABLE {t('govmove_postcode_centroids')}
CLUSTER BY (postcode)
COMMENT 'GovMove postcode lookup: mean British National Grid (EPSG:27700) Easting/Northing per postcode from addressbasepremium.'
AS
WITH base AS (
  SELECT upper(trim(postcode)) AS postcode, CAST(easting AS DOUBLE) AS e, CAST(northing AS DOUBLE) AS n,
         CAST(latitude AS DOUBLE) AS lat, CAST(longitude AS DOUBLE) AS lon,
         (effectiveenddate IS NULL AND addressstatus IN ('Approved', 'Provisional')) AS is_current
  FROM {t('addressbasepremium')}
  WHERE postcode IS NOT NULL AND easting > 0 AND northing > 0 AND easting <= 700000 AND northing <= 1300000
),
pc AS (SELECT postcode, max(is_current) AS has_current FROM base GROUP BY postcode)
SELECT b.postcode, avg(b.e) AS easting, avg(b.n) AS northing, avg(b.lat) AS latitude, avg(b.lon) AS longitude,
       count(*) AS address_count, p.has_current AS from_current_addresses, 'EPSG:27700' AS coordinate_reference_system
FROM base b JOIN pc p ON b.postcode = p.postcode AND b.is_current = p.has_current
GROUP BY b.postcode, p.has_current
""")
display(spark.sql(f"SELECT count(*) AS postcodes, count_if(NOT from_current_addresses) AS historical_only FROM {t('govmove_postcode_centroids')}"))

In [0]:
# One row per Insite Property Reference with BNG location and related-record counts.
if PRIORITY_TABLE and spark.catalog.tableExists(PRIORITY_TABLE):
    pq = ".".join(f"`{p}`" for p in PRIORITY_TABLE.split("."))
    priority_cte = (f"SELECT insite_property_reference AS property_reference, investigation_priority_score, priority_band, "
                    f"priority_explanation FROM {pq}")
else:
    priority_cte = ("SELECT CAST(NULL AS STRING) AS property_reference, CAST(NULL AS DOUBLE) AS investigation_priority_score, "
                    "CAST(NULL AS STRING) AS priority_band, CAST(NULL AS STRING) AS priority_explanation WHERE false")
    print("No priority_table supplied: priority score columns will be empty.")

spark.sql(f"""
CREATE OR REPLACE TABLE {t('govmove_property_evidence')}
COMMENT 'GovMove: one row per Insite Property Reference with BNG location and related-record counts.'
AS
WITH p AS (
  SELECT insite_property_reference AS property_reference,
         first(property_name_clean, true) AS property_name, first(full_address_clean, true) AS full_address,
         first(town_clean, true) AS town, first(region_clean, true) AS region, first(local_authority_clean, true) AS local_authority,
         first(postcode_clean, true) AS postcode, first(uk_nation, true) AS uk_nation,
         first(contract_parent_organisation, true) AS holding_department, first(asset_type, true) AS asset_type,
         first(usage, true) AS usage, first(tenure, true) AS tenure, count(DISTINCT insite_contract_reference) AS contracts,
         first(easting_clean, true) AS bng_easting, first(northing_clean, true) AS bng_northing,
         first(latitude_clean, true) AS latitude, first(longitude_clean, true) AS longitude
  FROM {t('govmove_property_clean')}
  WHERE insite_property_reference IS NOT NULL AND NOT is_duplicate_copy
  GROUP BY insite_property_reference),
b AS (SELECT insite_property_reference AS property_reference, count(*) AS buildings, sum(floor_area) AS building_floor_area
      FROM {t('govmove_building_clean')} WHERE NOT is_duplicate_copy GROUP BY 1),
l AS (SELECT insite_property_reference AS property_reference, count(*) AS land_records, sum(land_area_hectares) AS land_area_hectares
      FROM {t('govmove_land_clean')} WHERE NOT is_duplicate_copy GROUP BY 1),
o AS (SELECT insite_property_reference AS property_reference, count(*) AS occupation_records, count(DISTINCT organisation) AS occupiers,
             sum(floor_area) AS occupied_area, array_join(slice(array_sort(collect_set(organisation)), 1, 8), '; ') AS occupier_names
      FROM {t('govmove_occupation_clean')} WHERE NOT is_duplicate_copy GROUP BY 1),
v AS (SELECT property_reference, count(*) AS vacant_records, count_if(availability_status = 'Available') AS vacant_available_records,
             sum(CASE WHEN availability_status = 'Available' THEN available_area_sqm END) AS vacant_available_area
      FROM {t('govmove_property_locations')} WHERE dataset = 'vacant_space' GROUP BY 1),
s AS (SELECT property_reference, count(*) AS surplus_records, sum(available_area_sqm) AS surplus_floor_area,
             sum(surplus_land_area_hectares) AS surplus_land_area_hectares
      FROM {t('govmove_property_locations')} WHERE dataset = 'surplus' GROUP BY 1),
r AS ({priority_cte})
SELECT p.*, coalesce(b.buildings, 0) AS buildings, b.building_floor_area, coalesce(l.land_records, 0) AS land_records, l.land_area_hectares,
       coalesce(o.occupation_records, 0) AS occupation_records, coalesce(o.occupiers, 0) AS occupiers, o.occupied_area, o.occupier_names,
       coalesce(v.vacant_records, 0) AS vacant_records, coalesce(v.vacant_available_records, 0) AS vacant_available_records, v.vacant_available_area,
       coalesce(s.surplus_records, 0) AS surplus_records, s.surplus_floor_area, s.surplus_land_area_hectares,
       r.investigation_priority_score, r.priority_band, r.priority_explanation, p.bng_easting IS NOT NULL AS has_bng
FROM p LEFT JOIN b USING (property_reference) LEFT JOIN l USING (property_reference) LEFT JOIN o USING (property_reference)
LEFT JOIN v USING (property_reference) LEFT JOIN s USING (property_reference) LEFT JOIN r USING (property_reference)
""")
display(spark.sql(f"SELECT count(*) AS properties, count_if(has_bng) AS with_bng, count_if(occupiers > 1) AS multi_occupier, "
                  f"count(investigation_priority_score) AS with_priority_score FROM {t('govmove_property_evidence')}"))

In [0]:
objs = ["property", "building", "land", "occupation", "surplus", "vacant_space", "addressbasepremium", "govmove_property_clean",
        "govmove_building_clean", "govmove_land_clean", "govmove_occupation_clean", "vw_govmove_property_locations",
        "govmove_property_locations", "govmove_postcode_centroids", "govmove_property_evidence"]
display(spark.createDataFrame([(o, exists(o)) for o in objs], "object string, present boolean"))
print(f"""Set these in govmove/app.yaml, then grant the app's service principal access:
  GOVMOVE_CATALOG = {CATALOG}
  GOVMOVE_SCHEMA  = {SCHEMA}
  GOVMOVE_PRIORITY_TABLE = {PRIORITY_TABLE or '(leave unset)'}

  GRANT USE CATALOG ON CATALOG `{CATALOG}` TO `<app-service-principal-id>`;
  GRANT USE SCHEMA, SELECT ON SCHEMA `{CATALOG}`.`{SCHEMA}` TO `<app-service-principal-id>`;""")